# Guided Lab: From Channel Quality to Throughput Prediction

**Machine Learning for Communication Systems**  
This notebook develops a throughput predictor in small, interpretable steps. We begin with Shannon's capacity formula, calibrate it using linear regression, add radio-resource features, and finally compare random and chronological evaluation.

## Learning outcomes

By the end of the lab, students should be able to:

1. Convert SNR from dB to linear scale and construct a capacity proxy.
2. Explain why measured throughput differs from ideal channel capacity.
3. Fit and interpret simple regression models.
4. Prevent preprocessing and evaluation leakage.
5. Compare random and chronological train/test splits.
6. Diagnose model errors across radio operating conditions.

> The notebook deliberately avoids deep learning. The main tools are communications formulas, regression, and an optional clustering extension.

## 0. Setup

Place this notebook either inside `Explainable-AI-for-Telecom/` or in the repository root. The data-loading cell checks both locations. If the packages are missing, uncomment the installation line.

In [ ]:
# Uncomment only if your environment does not already contain these packages.
# %pip install pandas numpy matplotlib scikit-learn

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore', category=FutureWarning)

## 1. Load and inspect the measurements

The target is measured downlink throughput in bit/s. The inputs contain primary- and secondary-cell radio measurements, resource-allocation variables, vehicle context, weather, traffic, and location information.

In [ ]:
candidate_paths = [
    Path('Train.csv'),
    Path('Explainable-AI-for-Telecom/Train.csv'),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Train.csv was not found. Place the notebook in the repository root '
        'or inside Explainable-AI-for-Telecom/.'
    )

df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df['throughput_Mbps'] = df['target'] / 1e6

print(f'Data file: {DATA_PATH.resolve()}')
print(f'Rows: {len(df):,}; columns: {df.shape[1]}')
print(f'Time span: {df.datetime.min()} to {df.datetime.max()}')
df.head()

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_%': 100 * df.isna().mean(),
    'unique_values': df.nunique(dropna=True),
}).sort_values('missing_%', ascending=False)
summary.head(15).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['throughput_Mbps'], bins=60, color='steelblue', edgecolor='white')
axes[0].set(xlabel='Measured throughput [Mbit/s]', ylabel='Count', title='Throughput distribution')

axes[1].scatter(df['PCell_SNR_max'], df['throughput_Mbps'], s=5, alpha=0.15)
axes[1].set(xlabel='Primary-cell SNR [dB]', ylabel='Throughput [Mbit/s]', title='SNR alone does not determine throughput')
plt.tight_layout()
plt.show()

df['throughput_Mbps'].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])

### Checkpoint 1

Discuss before continuing:

1. Why can two samples with the same SNR have different throughput?
2. Which variables represent channel quality, and which represent resource allocation?
3. Why are many secondary-cell values missing? Is missingness necessarily a data error?
4. Why might GPS, timestamp, or cell identity give high predictive accuracy without teaching us a transferable radio relationship?

## 2. Construct a communications-based capacity proxy

Convert SNR from dB to linear scale:

$$\gamma = 10^{\mathrm{SNR}_{\mathrm{dB}}/10}.$$

Using bandwidth $B$ in MHz gives a capacity proxy in Mbit/s:

$$C = B\log_2(1+\gamma).$$

For carrier aggregation, add the primary- and secondary-cell proxies:

$$C_{\mathrm{total}}=C_p+I_s C_s,$$

where $I_s=1$ when a secondary cell is present. This is not an exact achievable-rate calculation: it ignores scheduling fraction, protocol overhead, coding loss, interference dynamics, spatial layers, and implementation constraints. It is an interpretable baseline.

In [ ]:
def snr_db_to_linear(snr_db):
    return 10.0 ** (snr_db / 10.0)

def capacity_proxy_mbps(bandwidth_mhz, snr_db):
    return bandwidth_mhz * np.log2(1.0 + snr_db_to_linear(snr_db))

df['secondary_present'] = (
    df['SCell_SNR_max'].notna() & df['SCell_Downlink_bandwidth_MHz'].notna()
).astype(int)

df['capacity_primary_Mbps'] = capacity_proxy_mbps(
    df['PCell_Downlink_bandwidth_MHz'], df['PCell_SNR_max']
)
df['capacity_secondary_Mbps'] = capacity_proxy_mbps(
    df['SCell_Downlink_bandwidth_MHz'], df['SCell_SNR_max']
).fillna(0.0)
df['capacity_total_Mbps'] = (
    df['capacity_primary_Mbps'].fillna(0.0)
    + df['secondary_present'] * df['capacity_secondary_Mbps']
)

df[[
    'throughput_Mbps', 'capacity_primary_Mbps',
    'capacity_secondary_Mbps', 'capacity_total_Mbps',
    'secondary_present'
]].describe().round(2)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['capacity_total_Mbps'], df['throughput_Mbps'], s=6, alpha=0.15)
limit = np.nanpercentile(df['capacity_total_Mbps'], 99.5)
plt.xlim(0, limit)
plt.xlabel('Shannon-inspired capacity proxy [Mbit/s]')
plt.ylabel('Measured throughput [Mbit/s]')
plt.title('Idealized capacity versus measured throughput')
plt.grid(alpha=0.2)
plt.show()

### Checkpoint 2

1. Should every point lie below the plotted proxy? Why might it not?
2. Is `PCell_Downlink_bandwidth_MHz` the same as the bandwidth actually allocated to this user?
3. Which omitted factors could explain large residual errors?

## 3. Create honest chronological splits

A random split mixes nearby measurements from the same drive into training and testing. We therefore use the earliest 60% for training, the next 20% for validation, and the latest 20% for final testing. All preprocessing must be fitted on the training portion only.

In [ ]:
df_time = df.sort_values(['timestamp', 'device']).reset_index(drop=True)
n = len(df_time)
train_end = int(0.60 * n)
valid_end = int(0.80 * n)

train_df = df_time.iloc[:train_end].copy()
valid_df = df_time.iloc[train_end:valid_end].copy()
test_df = df_time.iloc[valid_end:].copy()

split_summary = pd.DataFrame({
    'rows': [len(train_df), len(valid_df), len(test_df)],
    'start': [train_df.datetime.min(), valid_df.datetime.min(), test_df.datetime.min()],
    'end': [train_df.datetime.max(), valid_df.datetime.max(), test_df.datetime.max()],
}, index=['train', 'validation', 'test'])
split_summary

## 4. Evaluation helpers

We report

$$\mathrm{MAE}=\frac{1}{n}\sum_i|T_i-\widehat T_i|,$$

$$\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i(T_i-\widehat T_i)^2},$$

and

$$R^2=1-\frac{\sum_i(T_i-\widehat T_i)^2}{\sum_i(T_i-\overline T)^2}.$$

Ordinary MAPE is avoided because some measured throughput values are close to zero.

In [ ]:
TARGET = 'throughput_Mbps'

def regression_metrics(y_true, y_pred):
    return {
        'MAE [Mbit/s]': mean_absolute_error(y_true, y_pred),
        'RMSE [Mbit/s]': mean_squared_error(y_true, y_pred) ** 0.5,
        'R2': r2_score(y_true, y_pred),
    }

def evaluate_fitted_model(name, model, frame, features):
    prediction = np.maximum(0.0, model.predict(frame[features]))
    return name, regression_metrics(frame[TARGET], prediction), prediction

## 5. Baseline 0: Predict the training mean

A useful ML model must outperform a predictor that always returns the average training throughput.

In [ ]:
mean_model = DummyRegressor(strategy='mean')
mean_model.fit(np.zeros((len(train_df), 1)), train_df[TARGET])
mean_prediction = mean_model.predict(np.zeros((len(test_df), 1)))

results = {
    'Training mean': regression_metrics(test_df[TARGET], mean_prediction)
}
pd.DataFrame(results).T.round(3)

## 6. Baseline 1: Calibrate the capacity formula

Fit

$$\widehat T=\beta_0+\beta_1C_{\mathrm{total}}.$$

The slope is an empirical calibration factor, not a universal physical efficiency.

In [ ]:
capacity_features = ['capacity_total_Mbps']
capacity_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', LinearRegression()),
])
capacity_model.fit(train_df[capacity_features], train_df[TARGET])

name, metrics, capacity_prediction = evaluate_fitted_model(
    'Calibrated capacity', capacity_model, test_df, capacity_features
)
results[name] = metrics

reg = capacity_model.named_steps['regressor']
print(f'Intercept beta_0: {reg.intercept_:.3f} Mbit/s')
print(f'Capacity slope beta_1: {reg.coef_[0]:.3f}')
pd.DataFrame(results).T.round(3)

## 7. Baseline 2: Radio-aware multivariable regression

We now add channel quality, resource allocation, carrier aggregation, and speed. We intentionally exclude ID, raw timestamp, GPS, cell identity, and operator from the introductory model because they can act as fingerprints.

The numerical pipeline performs median imputation and standardization using training data only. `area` is one-hot encoded because it has no meaningful numerical order.

In [ ]:
for frame in [train_df, valid_df, test_df]:
    frame['log1p_PCell_RBs'] = np.log1p(frame['PCell_Downlink_Num_RBs'].clip(lower=0))
    frame['log1p_SCell_RBs'] = np.log1p(frame['SCell_Downlink_Num_RBs'].clip(lower=0))

numeric_features = [
    'capacity_total_Mbps',
    'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_SNR_max',
    'log1p_PCell_RBs', 'PCell_Downlink_Average_MCS',
    'SCell_RSRP_max', 'SCell_RSRQ_max', 'SCell_SNR_max',
    'log1p_SCell_RBs', 'SCell_Downlink_Average_MCS',
    'secondary_present', 'speed_kmh',
]
categorical_features = ['area']
model_features = numeric_features + categorical_features

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

radio_linear_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])
radio_linear_model.fit(train_df[model_features], train_df[TARGET])

name, metrics, radio_prediction = evaluate_fitted_model(
    'Radio-aware linear regression', radio_linear_model, test_df, model_features
)
results[name] = metrics
pd.DataFrame(results).T.round(3).sort_values('MAE [Mbit/s]')

## 8. Interpret the standardized regression coefficients

For standardized numerical inputs, a positive coefficient means that increasing the feature by one training-set standard deviation increases predicted throughput, holding the other transformed inputs fixed. This is an association—not necessarily a causal effect. Correlated radio variables can make individual coefficients unstable.

In [ ]:
feature_names = radio_linear_model.named_steps['preprocessor'].get_feature_names_out()
coefficients = radio_linear_model.named_steps['regressor'].coef_
coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
})
coef_table['absolute_coefficient'] = coef_table['coefficient'].abs()
coef_table.sort_values('absolute_coefficient', ascending=False).head(15)

### Checkpoint 3

1. Which variables have the largest positive and negative coefficients?
2. Are the signs consistent with communications intuition?
3. Why should we not interpret a regression coefficient as the effect of changing that variable in the network?
4. What happens to coefficients when two inputs contain similar information?

## 9. Inspect predictions and residuals

The residual is

$$e_i=T_i-\widehat T_i.$$

Positive residuals are under-predictions; negative residuals are over-predictions.

In [ ]:
analysis_test = test_df[[
    'throughput_Mbps', 'area', 'secondary_present',
    'PCell_SNR_max', 'datetime'
]].copy()
analysis_test['capacity_prediction'] = capacity_prediction
analysis_test['radio_prediction'] = radio_prediction
analysis_test['residual'] = analysis_test['throughput_Mbps'] - analysis_test['radio_prediction']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(analysis_test['throughput_Mbps'], analysis_test['radio_prediction'], s=7, alpha=0.2)
max_value = max(analysis_test['throughput_Mbps'].max(), analysis_test['radio_prediction'].max())
axes[0].plot([0, max_value], [0, max_value], '--', color='black', linewidth=1)
axes[0].set(xlabel='Measured [Mbit/s]', ylabel='Predicted [Mbit/s]', title='Measured versus predicted')

axes[1].scatter(analysis_test['radio_prediction'], analysis_test['residual'], s=7, alpha=0.2)
axes[1].axhline(0, linestyle='--', color='black', linewidth=1)
axes[1].set(xlabel='Predicted [Mbit/s]', ylabel='Residual [Mbit/s]', title='Residual diagnostic')
plt.tight_layout()
plt.show()

In [ ]:
def grouped_error(frame, group_column):
    return (
        frame.assign(absolute_error=frame['residual'].abs())
        .groupby(group_column, dropna=False)
        .agg(samples=('absolute_error', 'size'),
             MAE_Mbps=('absolute_error', 'mean'),
             mean_residual_Mbps=('residual', 'mean'))
        .sort_values('MAE_Mbps', ascending=False)
    )

display(grouped_error(analysis_test, 'area').round(3))
display(grouped_error(analysis_test, 'secondary_present').round(3))

## 10. Why random splitting can be misleading

Repeat the radio-aware regression with a random 80/20 split. This is not the final evaluation; it is an experiment showing how the evaluation protocol changes the apparent result.

In [ ]:
random_train, random_test = train_test_split(
    df_time.copy(), test_size=0.20, random_state=RANDOM_STATE
)
for frame in [random_train, random_test]:
    frame['log1p_PCell_RBs'] = np.log1p(frame['PCell_Downlink_Num_RBs'].clip(lower=0))
    frame['log1p_SCell_RBs'] = np.log1p(frame['SCell_Downlink_Num_RBs'].clip(lower=0))

random_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])
random_model.fit(random_train[model_features], random_train[TARGET])
random_prediction = np.maximum(0.0, random_model.predict(random_test[model_features]))

split_comparison = pd.DataFrame({
    'Chronological test': regression_metrics(test_df[TARGET], radio_prediction),
    'Random test': regression_metrics(random_test[TARGET], random_prediction),
}).T
split_comparison.round(3)

### Checkpoint 4

1. Which split gives better performance?
2. Does better random-split performance prove that the model will generalize to a future drive?
3. What kinds of temporal or spatial similarity could appear in both sides of a random split?
4. How could you design a cell-aware or route-aware split?

## 11. Optional extension: Discover operating regimes with clustering

Cluster input conditions—not throughput—using standardized radio features. Choose $K$ using the silhouette score

$$s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}.$$

Here $a(i)$ is the average distance to observations in the same cluster and $b(i)$ is the average distance to the nearest alternative cluster. The scaler and cluster centroids are fitted on training data only.

In [ ]:
cluster_features = [
    'PCell_SNR_max', 'PCell_RSRP_max',
    'PCell_Downlink_Average_MCS', 'log1p_PCell_RBs',
    'speed_kmh', 'secondary_present',
]
cluster_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
U_train = cluster_preprocessor.fit_transform(train_df[cluster_features])

silhouette_results = {}
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = candidate.fit_predict(U_train)
    silhouette_results[k] = silhouette_score(
        U_train, labels, sample_size=min(5000, len(U_train)), random_state=RANDOM_STATE
    )

pd.Series(silhouette_results, name='silhouette_score').rename_axis('K')

In [ ]:
best_k = max(silhouette_results, key=silhouette_results.get)
kmeans = KMeans(n_clusters=best_k, n_init=20, random_state=RANDOM_STATE)
train_clusters = kmeans.fit_predict(U_train)
test_clusters = kmeans.predict(cluster_preprocessor.transform(test_df[cluster_features]))

clustered_train = train_df.copy()
clustered_test = test_df.copy()
clustered_train['cluster'] = train_clusters
clustered_test['cluster'] = test_clusters

cluster_profile = clustered_train.groupby('cluster')[
    cluster_features + ['throughput_Mbps']
].mean().round(2)
print(f'Selected K = {best_k}')
cluster_profile

### Clustering questions

1. Give each cluster an engineering name based on its center.
2. Which cluster has the largest mean throughput?
3. Is a high-SNR cluster always a high-throughput cluster?
4. Compare the regression MAE within each cluster.
5. As a homework extension, fit a separate linear regression in each cluster and compare it with the global regression.

## 12. Suggested student deliverables

Submit a short report containing:

1. A table comparing the training mean, calibrated capacity, and radio-aware regression.
2. Random-split and chronological-split results.
3. One measured-versus-predicted plot and one residual plot.
4. An engineering interpretation of the five largest standardized coefficients.
5. One example where good SNR does not produce high throughput, with a proposed explanation.
6. A description of the discovered clusters, if the extension is completed.
7. A paragraph distinguishing predictive association from causal network intervention.

## Takeaway

A communications formula provides structure and interpretability; regression learns systematic departures from its assumptions. Reliable evaluation requires more than a high random-split score.